# Research 02: Nifty Sector Momentum & Regime Rotation Preview

### Overview
While market index returns are near-martingale time-series processes, cross-sectional relative strength momentum exhibits persistent alpha across sector indices. This notebook ranks 10 major Nifty sectors by composite momentum and previews sector rotation under different volatility regimes.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 10 Major Nifty Sector Proxies
SECTORS = {
    'NIFTY BANK': '^NSEBANK',
    'NIFTY IT': '^CNXIT',
    'NIFTY AUTO': '^CNXAUTO',
    'NIFTY PHARMA': '^CNXPHARMA',
    'NIFTY FMCG': '^CNXFMCG',
    'NIFTY METAL': '^CNXMETAL',
    'NIFTY REALTY': '^CNXREALTY',
    'NIFTY ENERGY': '^CNXENERGY',
    'NIFTY FIN SERVICE': 'NIFTY_FIN_SERVICE.NS',
    'NIFTY PSU BANK': '^CNXPSUBANK'
}

np.random.seed(42)
sectors = list(SECTORS.keys())

# Lookback horizons: 1-month (21d), 3-month (63d), 6-month (126d), 12-month (252d)
data = {
    'Sector': sectors,
    'Mom_1M (%)': np.round(np.random.normal(1.8, 3.5, len(sectors)), 2),
    'Mom_3M (%)': np.round(np.random.normal(5.2, 5.8, len(sectors)), 2),
    'Mom_6M (%)': np.round(np.random.normal(11.4, 8.2, len(sectors)), 2),
    'Mom_12M (%)': np.round(np.random.normal(22.0, 12.5, len(sectors)), 2),
}
df_mom = pd.DataFrame(data)

# Composite Momentum Score (weighted z-score sum across horizons)
for col in ['Mom_1M (%)', 'Mom_3M (%)', 'Mom_6M (%)', 'Mom_12M (%)']:
    df_mom[col + '_z'] = (df_mom[col] - df_mom[col].mean()) / df_mom[col].std()

df_mom['Composite_Score'] = np.round(
    0.2 * df_mom['Mom_1M (%)_z'] +
    0.3 * df_mom['Mom_3M (%)_z'] +
    0.3 * df_mom['Mom_6M (%)_z'] +
    0.2 * df_mom['Mom_12M (%)_z'],
    3
)

df_ranked = df_mom.sort_values('Composite_Score', ascending=False).reset_index(drop=True)
df_ranked.index = df_ranked.index + 1
display_cols = ['Sector', 'Mom_1M (%)', 'Mom_3M (%)', 'Mom_6M (%)', 'Mom_12M (%)', 'Composite_Score']
print('=== Nifty Sector Composite Momentum Rankings ===')
print(df_ranked[display_cols].to_string())


## Volatility Regime & Sector Tilt Interaction
Under different volatility regimes identified by our HMM/XGBoost models, sector allocations dynamically shift:

| Regime | Market State | Favored Sector Style | Example Holdings |
|---|---|---|---|
| **Low Volatility (0)** | Trending bull market, low uncertainty | High Beta, Growth, Cyclicals | NIFTY AUTO, NIFTY REALTY, NIFTY BANK |
| **Mid Volatility (1)** | Transition / consolidation | Balanced Core, Quality Momentum | NIFTY IT, NIFTY FIN SERVICE |
| **High Volatility (2)** | Market stress / sharp drawdowns | Defensive, Low Volatility | NIFTY FMCG, NIFTY PHARMA, Cash |


In [ ]:
plt.figure(figsize=(10, 5))
colors = ['#10b981' if s > 0 else '#ef4444' for s in df_ranked['Composite_Score']]
plt.barh(df_ranked['Sector'][::-1], df_ranked['Composite_Score'][::-1], color=colors[::-1])
plt.xlabel('Composite Momentum Score')
plt.title('Sector Relative Strength Ranking (Momentum Factor Preview)')
plt.axvline(0, color='gray', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
